# MedGuard AI - Data Cleaning & Feature Engineering

In this notebook, we will clean the dataset step-by-step according to the provided instructions:

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../data/raw/KaggleV2-May-2016.csv')
display(df.head())

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


### Step 1: Check and Remove Duplicates
We need to ensure there are no duplicate rows that could bias our machine learning model.

In [2]:
# Check for duplicate records
duplicates = df.duplicated().sum()
print(f"Number of duplicate records found: {duplicates}")

# If duplicates exist, we drop them (though typically this dataset has 0 perfect row duplicates)
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")

Number of duplicate records found: 0


### Step 2: Rename Columns
Some column names have typos (like `Hipertension` and `Handcap`). We will fix them to be consistent and readable.

In [3]:
df = df.rename(columns={
    'Hipertension': 'Hypertension',
    'Handcap': 'Handicap',
    'PatientId': 'PatientID',
    'No-show': 'No_Show'
})

print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['PatientID', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'No_Show']


### Step 3: Convert Dates to Datetime Format
The `ScheduledDay` and `AppointmentDay` are currently stored as strings (objects). We need to convert them to actual datetime objects so we can perform math on them.

In [4]:
# Convert to datetime and normalize to just the Date (strip the time of day)
# We strip the time because AppointmentDay has its time set to 00:00:00, 
# which makes exact time-level subtractions inaccurate.
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay']).dt.normalize()
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay']).dt.normalize()

print(df[['ScheduledDay', 'AppointmentDay']].dtypes)

ScheduledDay      datetime64[us, UTC]
AppointmentDay    datetime64[us, UTC]
dtype: object


### Step 4: Create 'WaitingDays' Feature
The difference in days between when the patient scheduled the appointment and the actual appointment day is a critical predictive feature. Patients who wait longer are generally more likely to no-show.

In [5]:
# Calculate WaitingDays
df['WaitingDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

# IMPORTANT DATA CLEANING CHECK: 
# Let's verify if there are any negative WaitingDays (which would mean the appointment happened before it was scheduled!)
invalid_dates = (df['WaitingDays'] < 0).sum()
print(f"Number of records with negative waiting days: {invalid_dates}")

# Drop these erroneous records
df = df[df['WaitingDays'] >= 0]
print("Dropped erroneous records.")

Number of records with negative waiting days: 5
Dropped erroneous records.


### Step 5: Encode the Target Column
Machine learning models prefer numbers, not text. We need to convert `No_Show` from 'Yes'/'No' into a binary label where:
* `1` = No-Show (They missed it - Non-Adherent)
* `0` = Show (They attended - Adherent)

In [6]:
df['No_Show'] = df['No_Show'].map({'Yes': 1, 'No': 0})

print("Target values after mapping:")
print(df['No_Show'].value_counts())

Target values after mapping:
No_Show
0    88208
1    22314
Name: count, dtype: int64


### Step 6: Save the Cleaned Dataset
We will save this refined dataframe into our `data/processed` folder so our ML models can ingest it directly without repeating these steps.

In [7]:
output_path = '../data/processed/cleaned_appointments.csv'
df.to_csv(output_path, index=False)
print(f"Successfully saved cleaned data to {output_path}")

Successfully saved cleaned data to ../data/processed/cleaned_appointments.csv
